In [ ]:
# Imports for auto-reloading
%load_ext autoreload
%autoreload 2

# Configure pandas to display all columns and rows
import pandas as pd

pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [ ]:
# Optionally force tensorflow on CPU
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Table of Contents
1. [General Usage](#general-usage)
    1. [Basic Usage](#basic-usage)
    2. [Advanced Usage](#advanced-usage)
2. [Data Generation](#data-gen)
    1. [Model Synthesis](#model-synth)
        1. [Keras Synthesis](#keras-synth)
        2. [PyTorch Synthesis](#torch-synth)
    2. [Parallel Synthesis](#parallel-synth)
        1. [Randomly Generated Networks](#random-synth)
3. [Training Prediction Models](#train-models)
    1. [Preparing a Dataset](#prepare-data)
    2. [Training an MLP](#train-mlp)
    3. [Training a GNN](#train-gnn)
4. [Testing Prediction Models](#test-models)
    1. [Metrics per Split](#metrics)
    2. [Predicted vs. Synthesized](#plots)

# 1. General Usage <a class="anchor" id="general-usage"></a>

This section shows both basic and advanced cases of using `rule4ml` for FPGA resources/latency prediction.

## 1.1 Basic Usage <a class="anchor" id="basic-usage"></a>

First, we go through a simple example, using Keras functional API to build a target model and rule4ml `MultiModelWrapper` class for predictions.

In [ ]:
import keras

# Example of a keras Model we want to predict for
input_shape = (16,)
inputs = keras.layers.Input(shape=input_shape)
x = keras.layers.Dense(32, activation="relu")(inputs)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dense(32, activation="relu")(x)
outputs = keras.layers.Dense(5, activation="softmax")(x)

model_to_predict = keras.Model(inputs=inputs, outputs=outputs, name="Jet Classifier")
model_to_predict.build((None, *input_shape))

Calling `load_default_models` will load the weights included within the package for each resource and latency predictor.

Currently the predicted resources are BRAM, DSP, FF and LUT, while latency refers to the number of clock cycles required for an inference of the target model(s).

**Important:** Since the predictors are trained and their weights are saved using a specific TF/Keras version, using a lower version might break weight loading. In case weight loading fails, check if your installed version matches [setup.cfg](https://github.com/IMPETUS-UdeS/rule4ml/blob/main/setup.cfg).

In [ ]:
from rule4ml.models.wrappers import MultiModelWrapper

# Loading default weights
estimator = MultiModelWrapper()
estimator.load_default_models()

After building the target model and loading the predictors weights, what's left is to call `predict`.

In case the target models are the only argument passed to the `predict` method, predictions are made for hls4ml configurations seen during training.

Later on, we will see how to make a specific configuration we're interested in and pass it as an argument to `predict`.

In [ ]:
# MultiModelWrapper predictions are formatted as a DataFrame
prediction_df = estimator.predict(model_to_predict, verbose=1)

# index by model and settings
if not prediction_df.empty:
    prediction_df = prediction_df.set_index(
        [
            "Model",
            "Board",
            "Strategy",
            "Precision",
            "Reuse Factor",
            "HLS4ML Version",
            "Vivado Version",
        ]
    )

prediction_df

`MultiModelWrapper` returns a pandas `DataFrame`, giving access to many useful operations (min, max, groupby, where, etc.)

The prediction dataframe can be exported in various formats as well. We recommend saving as HTML.

In [ ]:
prediction_df.to_html("keras_example_v2_gnn.html")

# prediction_df.to_latex("keras_example.tex")
# prediction_df.to_csv("keras_example.csv")
# prediction_df.to_json("keras_example.json")
# prediction_df.to_xml("keras_example.xml")

## 1.2 Advanced Usage <a class="anchor" id="advanced-usage"></a>

Here, we explore alternative and more flexible ways to load weights and make predictions.

In [ ]:
import itertools

import keras
import torch

models_to_predict = []


# Example of a subclassed PyTorch model
class MyTopQuarks(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.dense1 = torch.nn.Linear(10, 32)
        self.relu = torch.nn.ReLU()
        self.dense2 = torch.nn.Linear(32, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, inputs):
        x = self.dense1(inputs)
        x = self.relu(x)
        x = self.dense2(x)
        outputs = self.sigmoid(x)

        return outputs


models_to_predict.append(MyTopQuarks())

# Example of a keras Sequential model
input_size = 16
model_to_predict = keras.Sequential(
    layers=[
        keras.layers.Input(shape=(input_size,)),
        keras.layers.Dense(32, use_bias=True),
        keras.layers.Activation("relu"),
        keras.layers.Dense(32, use_bias=True),
        keras.layers.Activation("relu"),
        keras.layers.Dense(32, use_bias=True),
        keras.layers.Activation("relu"),
        keras.layers.Dense(5, use_bias=True),
        keras.layers.Activation("softmax"),
    ],
    name="Jet Classifier",
)
model_to_predict.build((None, input_size))

models_to_predict.append(model_to_predict)

# Instead of default configs, we can specify custom configurations we want to predict for
hls_configs = [
    {
        "model": {
            "precision": "ap_fixed<8, 3>",
            "reuse_factor": 32,
            "strategy": strategy,
        },
        "board": board,
    }
    for board, strategy in itertools.product(
        ["pynq-z2", "zcu102"], ["Latency", "Resource"]
    )
]

With the `ModelWrapper`, it's possible to load individual predictors. Let's say we're only interested in predicting **LUT** this time, using an MLP-based estimator.

In [ ]:
from rule4ml.models.wrappers import KerasModelWrapper

lut_model_wrapper = KerasModelWrapper()
lut_model_wrapper.load(
    "../rule4ml/models/weights/v2/mlp/LUT.config.json",
    "../rule4ml/models/weights/v2/mlp/LUT.weights.h5",
)  # Load LUT predictor

lut_model_wrapper.predict(
    models_to_predict, hls_configs
)  # ModelWrapper returns an ndarray of predictions, one for each model/config combination

Alternatively, we can add the previous `ModelWrapper` to a new instance of `MultiModelWrapper` for a nicely formatted `DataFrame` output.

In [ ]:
from rule4ml.models.wrappers import MultiModelWrapper

estimator = MultiModelWrapper()
estimator.add_model_wrapper(lut_model_wrapper)

estimator.predict(models_to_predict, hls_configs=hls_configs)

# 2. Data Generation <a class="anchor" id="data-gen"></a>

In [ ]:
import os
import sys

# Specify Vivado path
os.environ["PATH"] = "/opt/Xilinx/Vivado/2019.1/bin:" + os.environ["PATH"]

base_path = os.path.join(os.getcwd(), "..", "data_gen")
sys.path.append(base_path)

## 2.1 Model Synthesis <a class="anchor" id="model-synth"></a>

### 2.1.1 Keras Model <a class="anchor" id="keras-synth"></a>

In [ ]:
import keras

from data_gen.nn_synth import synthesize_keras_model

input_size = 16
inputs = keras.layers.Input(shape=(input_size,))
x = keras.layers.Dense(32, activation="relu")(inputs)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dense(32, activation="relu")(x)
outputs = keras.layers.Dense(5, activation="softmax")(x)

model_to_synthesize = keras.Model(inputs=inputs, outputs=outputs, name="Jet Classifier")
model_to_synthesize.build((None, input_size))

synthesis_result = synthesize_keras_model(
    model_to_synthesize,
    board="pynq-z2",
    strategy="Resource",
    precision="ap_fixed<8, 3>",
    reuse_factor=32,
    clock_period="10",
    io_type="io_parallel",
    project_dir="./hls4ml_prj",
    synth_uuid=None,
    verbose=0,
)

In [ ]:
from data_gen.utils import save_to_json

save_to_json(synthesis_result, "./synthesis_result.json")

### 2.1.2 PyTorch Model <a class="anchor" id="torch-synth"></a>

In [ ]:
import torch

from data_gen.nn_synth import synthesize_torch_model
from data_gen.utils import save_to_json

model_to_synthesize = torch.nn.Sequential(
    torch.nn.Linear(10, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1),
    torch.nn.Sigmoid(),
)

synthesis_result = synthesize_torch_model(
    model_to_synthesize,
    board="zcu102",
    strategy="Latency",
    precision="ap_fixed<8, 3>",
    reuse_factor=32,
    clock_period="10",
    io_type="io_parallel",
    project_dir="./hls4ml_prj",
    synth_uuid=None,
    verbose=0,
)

save_to_json(synthesis_result, "./synthesis_result.json")

## 2.2 Parallel Synthesis <a class="anchor" id="parallel-synth"></a>

### 2.2.1 Randomly Generated Networks <a class="anchor" id="random-synth"></a>

In [ ]:
import time
from multiprocessing import Pool

from data_gen.nn_gen import (
    GeneratorSettings,
    IntRange,
    Power2Range,
    generate_fc_network,
)
from data_gen.nn_synth import (
    SynthSettings,
    parallel_generative_synthesis,
    synthesize_keras_model,
)
from data_gen.utils import save_to_json

gen_settings = GeneratorSettings(
    input_range=Power2Range(16, 32),
    layer_range=IntRange(2, 3),
    neuron_range=Power2Range(16, 32),
    output_range=IntRange(1, 20),
    activations=["relu"],
)
synth_settings = SynthSettings(
    reuse_range=Power2Range(32, 64),
    precisions=["ap_fixed<2, 1>", "ap_fixed<8, 3>"],
    strategies=["Resource"],
)

n_procs = 3
with Pool(n_procs) as p:
    result = p.map_async(
        parallel_generative_synthesis,
        [
            {
                "job_id": f"{proc}",
                "n_models": 10,
                "project_dir": "./projects",
                "prj_overwrite": False,
                "save_path": "./",
                "rng_seed": 0,
                "gen_function": generate_fc_network,  # Keras networks only currently
                "gen_settings": gen_settings,
                "synth_function": synthesize_keras_model,
                "synth_settings": synth_settings,
            }
            for proc in range(1, n_procs + 1)
        ],
    )
    while not result.ready():
        time.sleep(1)
    result = result.get()
    p.terminate()
    p.join()

# 3. Training Prediction Models <a class="anchor" id="train-models"></a>

## 3.1 Preparing a Dataset <a class="anchor" id="prepare-data"></a>

Synthesis records (JSON files, as produced by `data_gen` or downloaded from [wa-hls4ml](https://huggingface.co/datasets/fastmachinelearning/wa-hls4ml)) are parsed once into a typed `.npz` file.

The file holds every feature rule4ml extracts, plus raw targets from both resource reports (`resource_report` for logic synthesis, `hls_resource_report` for HLS estimates) and the latency report. Missing targets are stored as NaN.

In [ ]:
import glob
import os

from rule4ml import dataset

data_dir = os.path.join("..", "datasets", "wa-hls4ml")
train_path = os.path.join(data_dir, "train.npz")
if not os.path.exists(train_path):
    dataset.prepare(
        sorted(glob.glob(os.path.join(data_dir, "train", "*.json"))), train_path
    )

train_data, val_data = dataset.split(dataset.load(train_path), val_ratio=0.15, seed=0)
print(f"Train records: {len(train_data)}, validation records: {len(val_data)}")
print(f"Layer features: {train_data.layer_names}")

Prepare the test split and the exemplar models the same way. This tutorial evaluates on the validation split.

## 3.2 Training an MLP <a class="anchor" id="train-mlp"></a>

A predictor is built from three choices:

- **Numerical features**, by name. Categorical inputs (strategy, board, ...) come from the model's category maps.
- **Targets**, any of `bram`, `dsp`, `ff`, `lut`, `cycles`, `interval`.
- **The resource report** the resource targets are read from.

Records missing a selected target are skipped during training. The model input width is the number of numerical features plus the number of categorical inputs.

In [ ]:
from rule4ml.models.architectures import KerasMLP, MLPSettings
from rule4ml.models.wrappers import KerasModelWrapper, TrainSettings
from rule4ml.parsers.data_parser import (
    default_board_map,
    default_layer_type_map,
    default_strategy_map,
)

global_maps = {"strategy": default_strategy_map, "board": default_board_map}
global_labels = [
    "bit_width",
    "integer_bits",
    "reuse_mean",
    "dense_count",
    "dense_inputs_mean",
    "dense_outputs_mean",
    "dense_parameters_mean",
    "dense_reuse_mean",
    "conv1d_count",
    "conv1d_inputs_mean",
    "conv1d_outputs_mean",
    "conv1d_parameters_mean",
    "relu_count",
    "sigmoid_count",
    "tanh_count",
    "softmax_count",
    "batchnormalization_count",
    "total_add",
    "total_mult",
    "total_logical",
    "total_lookup",
]
target_labels = ["lut"]

mlp_wrapper = KerasModelWrapper()
mlp_wrapper.set_model(
    KerasMLP(
        MLPSettings(
            embedding_layers=[8, 8],
            numerical_dense_layers=[64],
            dense_layers=[128, 64, 32],
            dense_dropouts=[0.0, 0.1, 0.0],
        ),
        input_shape=(None, len(global_labels) + len(global_maps)),
        output_shape=(None, len(target_labels)),
        categorical_maps=global_maps,
        name="LUT_MLP",
    ),
    global_labels,
    target_labels,
    report="resource_report",
)

train_settings = TrainSettings(
    num_epochs=20, batch_size=128, learning_rate=1e-3, loss_function="msle"
)
history = mlp_wrapper.fit(train_data, train_settings, val_dataset=val_data, verbose=2)

`evaluate` reports the wa-hls4ml metrics in raw units, per target, over the records where that target is known. SMAPE uses +1 in the denominator, as in the wa-hls4ml paper.

In [ ]:
mlp_wrapper.evaluate(val_data)

In [ ]:
mlp_wrapper.save("./models/lut_mlp")  # config JSON and weights, loadable with load()

## 3.3 Training a GNN <a class="anchor" id="train-gnn"></a>

Graph models also read per-layer features. Each record's layers become a chain graph. Padding is detected as all-zero layer features, so the selected layer features must not be zero for a real layer.

In [ ]:
from rule4ml.models.architectures import GNNSettings, TorchGNN
from rule4ml.models.callbacks import EarlyStopping
from rule4ml.models.wrappers import TorchModelWrapper

sequential_maps = {"layer_type": default_layer_type_map}
sequential_labels = [
    "layer_input_size",
    "layer_output_size",
    "layer_parameter_count",
    "layer_reuse",
    "layer_op_add",
    "layer_op_mult",
]

gnn_wrapper = TorchModelWrapper()
gnn_wrapper.set_model(
    TorchGNN(
        GNNSettings(
            numerical_dense_layers=[32],
            gconv_layers=[64, 64],
            global_embedding_layers=[8, 8],
            seq_embedding_layers=[8],
            dense_layers=[128, 64],
            bayesian_dense=[False, False],
            dense_dropouts=[0.1, 0.0],
        ),
        global_input_shape=(None, len(global_labels) + len(global_maps)),
        sequential_input_shape=(None, len(sequential_labels) + len(sequential_maps)),
        output_shape=(None, len(target_labels)),
        global_categorical_maps=global_maps,
        sequential_categorical_maps=sequential_maps,
        name="LUT_GNN",
    ),
    global_labels,
    target_labels,
    sequential_labels,
    report="resource_report",
)

history = gnn_wrapper.fit(
    train_data,
    train_settings,
    val_dataset=val_data,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5)],
    verbose=1,
)
gnn_wrapper.evaluate(val_data)

# 4. Testing Prediction Models <a class="anchor" id="test-models"></a>

## 4.1 Metrics per Split <a class="anchor" id="metrics"></a>

In [ ]:
import pandas as pd

splits = {"train": train_data, "validation": val_data}
scores = {
    (split, wrapper.model.name, label): score
    for split, data in splits.items()
    for wrapper in (mlp_wrapper, gnn_wrapper)
    for label, score in wrapper.evaluate(data).items()
}
pd.DataFrame(scores).T.round(2)

## 4.2 Predicted vs. Synthesized <a class="anchor" id="plots"></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

true = val_data.targets(target_labels, "resource_report")[:, 0]
known = np.isfinite(true)

fig, ax = plt.subplots(figsize=(6, 6))
for wrapper in (mlp_wrapper, gnn_wrapper):
    predicted = wrapper.predict_dataset(val_data)[:, 0]
    ax.scatter(true[known], predicted[known], s=4, alpha=0.3, label=wrapper.model.name)

limits = [1, max(true[known].max(), 10)]
ax.plot(limits, limits, "k--", linewidth=1)
ax.set(xscale="log", yscale="log", xlabel="Synthesized LUT", ylabel="Predicted LUT")
ax.legend()
plt.show()